## HAVEN-RVS: Manual Computation Synced Pipeline
This notebook trains a model to predict the Building Risk Index based on the structural manual computation reference.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

RANDOM_SEED = 42
SYNTHETIC_SAMPLES = 6000
np.random.seed(RANDOM_SEED)

# --- MAPPINGS ---
PEIS_MAP = {'VI': 2, 'VII': 3, 'VIII': 3, 'IX': 3, 'X': 3}
LIQ_MAP  = {'Safe': 1, 'Least Susceptible': 1, 'Moderately Susceptible': 2, 'Highly Susceptible': 3}

def compute_hazard_rating(row):
    a1 = (row['peis_s']*0.224) + (row['fault_s']*0.185) + (row['source_s']*0.364) + (row['liq_s']*0.227)
    a2 = (row['wind_s']*0.657) + (row['terrain_s']*0.343)
    a3 = (row['slope_s']*0.087) + (row['elev_s']*0.211) + (row['water_s']*0.175) + (row['runoff_s']*0.269) + (row['base_s']*0.14) + (row['drain_s']*0.118)
    return np.mean([a1, a2, a3])

def compute_exposure_rating(row):
    b1 = (row['b11']*0.159) + (row['b12']*0.168) + (row['b13']*0.344) + (row['b14']*0.329)
    b2 = (row['age_s']*0.401) + (row['b22']*0.125) + (row['b23']*0.093) + (row['b24']*0.158) + (row['b25']*0.223)
    b3 = (row['b31']*0.378) + (row['b32']*0.217) + (row['b33']*0.133) + (row['b34']*0.272)
    b4 = (row['b41']*0.244) + (row['b42']*0.361) + (row['b43']*0.115) + (row['b44']*0.28)
    return np.mean([b1, b2, b3, b4])

def compute_vulnerability_rating(row):
    c1_w = [0.092, 0.053, 0.057, 0.063, 0.031, 0.098, 0.051, 0.082, 0.146, 0.113, 0.102, 0.069]
    c1_s = [row[f'c1_{i}'] for i in range(12)]
    c1 = sum(s * w for s, w in zip(c1_s, c1_w))
    c2_w = [0.158, 0.147, 0.213, 0.124, 0.133, 0.225]
    c2_s = [row[f'c2_{i}'] for i in range(6)]
    c2 = sum(s * w for s, w in zip(c2_s, c2_w))
    c3 = (row['r_design']*0.344) + (row['r_slope']*0.424) + (row['r_mat']*0.232)
    c4 = (row['f_type']*0.632) + (row['f_dist']*0.368)
    return np.mean([c1, c2, c3, c4])

def compute_risk_index(row):
    H = compute_hazard_rating(row)
    E = compute_exposure_rating(row)
    V = compute_vulnerability_rating(row)
    return round(((H * E * V) / 27) * 10, 4)

def generate_data(n=6000):
    rows = []
    for _ in range(n):
        r = {}
        # Simplified: generating scores directly (1-3) to match manual calc logic
        for k in ['peis_s','fault_s','source_s','liq_s','wind_s','terrain_s','slope_s','elev_s','water_s','runoff_s','base_s','drain_s']: r[k] = np.random.randint(1,4)
        for k in ['b11','b12','b13','b14','b22','b23','b24','b25','b31','b32','b33','b34','b41','b42','b43','b44','age_s']: r[k] = np.random.randint(1,4)
        for i in range(12): r[f'c1_{i}'] = np.random.randint(1,4)
        for i in range(6): r[f'c2_{i}'] = np.random.randint(1,4)
        for k in ['r_design','r_slope','r_mat','f_type','f_dist']: r[k] = np.random.randint(1,4)
        r['risk_index'] = compute_risk_index(r)
        rows.append(r)
    return pd.DataFrame(rows)

print('Training Unified Model...')
data = generate_data()
X = data.drop('risk_index', axis=1)
y = data['risk_index']
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)
model.fit(X_scaled, y)

os.makedirs('model_exports', exist_ok=True)
pickle.dump(model, open('model_exports/best_model.pkl', 'wb'))
pickle.dump(scaler, open('model_exports/scaler.pkl', 'wb'))
# Dummy label encoder for compatibility
le = LabelEncoder().fit(['LOW RISK', 'MODERATE RISK', 'HIGH RISK'])
pickle.dump(le, open('model_exports/label_encoder.pkl', 'wb'))
print('✅ Model re-trained and exported.')